# 16. `running` (partition, status)

In [1]:
from common import *

shape, log = show(16)

log shape 81: partition | status | running
3,373 logs (0.8%) from 1 nodes, 2003-07-31 11:04:04 to 2006-04-27 14:58:33
log 78584 at 2004-01-01 05:30:36: 2557201 full partition status 1072935036 1 running


`full partition status running` isn't a problem probably, but let's see what happened before this. Does this follow an outage? Is the partition going up and down and not truly recovering? Or is it just noise, a message printed every X time?

What the full partition logged in the day before

In [2]:
day_before = timeline(log.node, log.ts - day, log.ts)
day_before

,first_ts,last_ts,count,component,state,template
0,2003-12-31 07:00:30,2003-12-31 07:00:30,1,partition,status,blocked
1,2003-12-31 07:00:35,2003-12-31 07:00:35,1,partition,status,running
2,2003-12-31 08:06:00,2003-12-31 08:06:00,1,partition,status,blocked
3,2003-12-31 08:06:05,2003-12-31 08:06:05,1,partition,status,running
4,2003-12-31 09:13:30,2003-12-31 09:13:30,1,partition,status,blocked
5,2003-12-31 09:13:35,2003-12-31 09:13:35,1,partition,status,running
6,2003-12-31 10:26:00,2003-12-31 10:26:00,1,partition,status,blocked
7,2003-12-31 10:26:05,2003-12-31 10:26:05,1,partition,status,running
8,2003-12-31 13:03:00,2003-12-31 13:03:00,1,partition,status,blocked
9,2003-12-31 13:03:04,2003-12-31 13:03:04,1,partition,status,running


So our message follows `blocked` quite regularly in the last day, and it seems the partition comes up very quickly each time, including this time
- Seconds from each `blocked` to the next line

In [3]:
changes = pd.to_datetime(day_before.first_ts)
seconds_blocked = changes.diff().shift(-1).dt.total_seconds()[day_before.template == "blocked"].dropna()
print(f"{len(seconds_blocked)} times blocked, for {seconds_blocked.min():.0f} to {seconds_blocked.max():.0f} seconds each time.")
assert seconds_blocked.max() < 60

13 times blocked, for 4 to 6 seconds each time.


The `blocked` part could be a problem or it could just be something normal happening when the system is under heavy load. The verdict for this specific `running` log is obviously non-threat, but let's continue investigating the recurring `blocked` messages.
What happened on a previous time it was blocked?
- Nodes that stop responding within 15 minutes of a previous block

In [4]:
prev_block = day_before[day_before.template == "blocked"].iloc[3]
down = logs.stats(log.ts - 15 * minute, log.ts + 15 * minute, log_shape_id=NOT_RESPONDING)
print(down)
assert down.n_logs == 0

LogStats(n_logs=0, n_nodes=0, first_seen=None, last_seen=None)


Not convincing enough.
- Nodes that had stopped responding anytime in the past day

In [5]:
logs.stats(log.ts - day, log.ts, log_shape_id=NOT_RESPONDING)

LogStats(n_logs=0, n_nodes=0, first_seen=None, last_seen=None)

Seems like the blocks don't really affect anything here

In [6]:
print("Verdict: a recovery, no action. The partition (the pool of nodes that runs user jobs) was 'blocked' for a few seconds and is running again, with no node in trouble.")
print("And the line to watch is the 'blocked' before it anyway, not this one.")

Verdict: a recovery, no action. The partition (the pool of nodes that runs user jobs) was 'blocked' for a few seconds and is running again, with no node in trouble.
And the line to watch is the 'blocked' before it anyway, not this one.
